## Scheduling Didymos Observations

In [2]:
from datetime import datetime, timedelta

import os
import sys
from pathlib import Path
import shutil

sys.path.insert(0, '/home/mwalker/git/neoexchange/neoexchange') #point to top of file path
os.environ.setdefault('DJANGO_SETTINGS_MODULE', 'neox.settings')
os.environ.setdefault('DJANGO_ALLOW_ASYNC_UNSAFE', 'true')

import django
django.setup()
from django.conf import settings


os.chdir("/home/mwalker/git/neoexchange/neoexchange") #temp fix for json issue

from django.contrib.auth.models import User

from core.models import StaticSource, Block, Frame
from core.views import schedule_submit, record_block
from astrometrics.site_config import inst_overhead



In [1]:
# Scheduling Didymos
site = 'coj'
username = 'tlister@lcogt.net'
user = User.objects.get(username=username)
proposal = 'LCO2026A-003'

# Only need to change these
to_schedule = True
obs_date = datetime(2026, 7, 11)
exptime = 210

num_exps = int((6*60) / ((exptime + inst_overhead['muscat_exp_overhead']) / 60.0) ) + 1
td = obs_date-datetime(2026, 7, 6) # Don't change this value
field_num = td.days
field = StaticSource.objects.get(name__contains=f'2026 Field #{field_num:02d}')
print(f"Scheduling Didymos on {obs_date.strftime('%Y-%m-%d')} for field: {field.name[-14:]}")
data = {'ra_deg' : field.ra, 'dec_deg' :  field.dec, 'source_id' : field.name,
        'slot_length' : (6*60)+5,
        'exp_count' : num_exps, 'exp_length' : exptime, 'gp_explength' : exptime,  'rp_explength' : exptime,  'ip_explength' : exptime,  'zp_explength' : exptime,
        'filter_pattern' : 'gp',
        'site' : site.lower(), 'site_code' : 'K92' if site == 'cpt' else 'E10' ,
        'start_time' : obs_date, 'end_time' : obs_date+timedelta(seconds=86400-1),
        'user_id' : username, 'group_name' : f'65803_E10_{obs_date.strftime("%Y%m%d")}',
        'proposal_code' : proposal, # 'LCOEngineering', #'MuSCAT Commissioning',
        'max_airmass' : 2.0, 'min_lunar_dist' : 30, #20,
        'too_mode' : True, 'bin_mode' : None, 'muscat_sync' : False, 'instrument_code' : '', 'period' : None, 'jitter' : None}
tracking_num = None
if to_schedule is True:
    print("Submitting to Scheduler")
    tracking_num, sched_params = schedule_submit(data, field, data['user_id'])
else:
    print(data)
if tracking_num is not None:
    try:
        block_resp = record_block(tracking_num, sched_params, data, field, user)
    except TypeError:
        block_resp = record_block(tracking_num, sched_params, data, field)
    print(f"Created Block {tracking_num:} ? {block_resp:}")

NameError: name 'User' is not defined